# InternScenes scene_id -> USD

Converts a scene into an Isaac Sim-compatible USD file given a `scene_id`, fetching only that scene's data.

`data/` (symlinked to `external-lib/InternScenes/data`) holds a curated, per-scene subset of the dataset: `Layout_info/` (scene metadata) and `asset_library/` (GLB assets). The full `InternRobotics/InternScenes` HF repo is ~2.46 TB and must never be mirrored in full -- see the download section below.

Run the cells in order; change `SCENE_ID` wherever it appears (e.g. `scene0000_01`, `scene0001_00`).

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path('/home/snt/projects/AgenticMemoryNav')
DATA_ROOT = PROJECT_ROOT / 'data'
# Curated per-scene subset (Layout_info/ + asset_library/) lives directly under DATA_ROOT.
# NOTE: DATA_ROOT / 'InternScenes' is a *separate*, full raw HF clone (100+ GB) -- never use that path here.
DATASET_ROOT = DATA_ROOT
INTERNSCENES_SRC = PROJECT_ROOT / 'external-lib' / 'InternScenes'
ISAAC_PY = Path('/home/snt/isaacsim/python.sh')
INTERNSCENES_VENV_PY = PROJECT_ROOT / '.internScenes-venv' / 'bin' / 'python3'

for p in [str(INTERNSCENES_SRC), str(INTERNSCENES_SRC / 'InternScenes')]:
    if p not in sys.path:
        sys.path.insert(0, p)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('DATASET_ROOT =', DATASET_ROOT)
print('INTERNSCENES_SRC =', INTERNSCENES_SRC)
print('ISAAC_PY exists =', ISAAC_PY.exists())
print('INTERNSCENES_VENV_PY exists =', INTERNSCENES_VENV_PY.exists())

print('layout folder exists =', (DATASET_ROOT / 'Layout_info').exists())
print('asset library exists =', (DATASET_ROOT / 'asset_library').exists())

PROJECT_ROOT = /home/snt/projects/AgenticMemoryNav
DATASET_ROOT = /home/snt/projects/AgenticMemoryNav/data
INTERNSCENES_SRC = /home/snt/projects/AgenticMemoryNav/external-lib/InternScenes
ISAAC_PY exists = True
INTERNSCENES_VENV_PY exists = True
layout folder exists = True
asset library exists = True


## Download only what a scene needs (recommended)

The full `InternRobotics/InternScenes` dataset is ~2.46 TB, so never `git clone` or `snapshot_download` the whole repo (the raw clone under `data/InternScenes/` alone is already 150+ GB on this machine and disk is nearly full). Instead, fetch data **per scene**:

1. **Layout metadata** -- `Layout_info.tar.gz` (~2.8 GB, covers *all* datasets/scenes) is downloaded once and cached under `data/.hf_layout/`. A single scene's `layout.json` + `StructureMesh/*.glb` are extracted from that local cache on demand, with **no extra network call**.
2. **3D asset GLBs** -- `download_scene_assets.py` reads a scene's `layout.json`, resolves only the unique `model_uid`s it references, and downloads just those files from Hugging Face (a typical scene needs ~10-100 small GLBs, tens to a few hundred MB, instead of the full multi-terabyte `asset_library`).

Run the next two cells for any `scene_id` before composing/converting it.

In [2]:
import subprocess
import tarfile

LAYOUT_TAR = DATA_ROOT / '.hf_layout' / 'Layout_info.tar.gz'
DOWNLOAD_ASSETS_SCRIPT = INTERNSCENES_SRC / 'download_scene_assets.py'


def ensure_layout_for_scene(scene_id: str, dataset: str = 'scannet') -> Path:
    """Extract one scene's layout.json/StructureMesh from the cached Layout_info tar (no network)."""
    target_dir = DATA_ROOT / 'Layout_info' / dataset / scene_id
    layout_path = target_dir / 'layout.json'
    if layout_path.exists():
        return layout_path

    if not LAYOUT_TAR.exists():
        raise FileNotFoundError(
            f'{LAYOUT_TAR} not found. Fetch it once (whole-file, ~2.8 GB, covers every scene) with:\n'
            f'  huggingface-cli download InternRobotics/InternScenes Layout_info.tar.gz '
            f'--repo-type dataset --local-dir {DATA_ROOT / ".hf_layout"}'
        )

    prefix = f'Layout_info/{dataset}/{scene_id}/'
    with tarfile.open(LAYOUT_TAR, 'r:gz') as tf:
        members = [m for m in tf.getmembers() if m.name.startswith(prefix)]
        if not members:
            raise FileNotFoundError(f'No entries for {prefix} inside {LAYOUT_TAR}. Check the scene id/dataset.')
        tf.extractall(path=DATA_ROOT, members=members)

    if not layout_path.exists():
        raise FileNotFoundError(f'Extraction finished but {layout_path} is still missing')
    return layout_path


def download_scene_assets_for_id(
    scene_id: str,
    dataset: str = 'scannet',
    no_archives: bool = False,
    with_objaverse: bool = False,
    dry_run: bool = False,
) -> subprocess.CompletedProcess:
    """Fetch only the GLB assets that one scene's layout.json references (per-scene, not the full library)."""
    layout_path = ensure_layout_for_scene(scene_id, dataset)
    cmd = [str(INTERNSCENES_VENV_PY), str(DOWNLOAD_ASSETS_SCRIPT), str(layout_path), '--dest', str(DATA_ROOT)]
    if no_archives:
        cmd.append('--no-archives')
    if with_objaverse:
        cmd.append('--with-objaverse')
    if dry_run:
        cmd.append('--dry-run')
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print('STDERR:\n', result.stderr[-2000:])
    return result


## Convert scene id(s) to USD

`convert_scene_id_to_usd` ensures the layout + assets are present (per-scene, via the helpers above), then runs the headless Isaac Sim converter on the composed GLB.

In [3]:
import subprocess
from pathlib import Path
from typing import Optional, Tuple

def compose_scene_id(scene_id: str, dataset: str = 'scannet') -> Path:
    """Compose one scene's layout.json + assets into a single GLB (via compose_one.py)."""
    intern_src = PROJECT_ROOT / 'external-lib' / 'InternScenes'
    glb_path = PROJECT_ROOT / 'tutorial' / 'examples' / 'composed_scenes' / dataset / scene_id / 'glb_scene.glb'
    if glb_path.exists():
        return glb_path

    cmd = [str(INTERNSCENES_VENV_PY), str(intern_src / 'compose_one.py'), f'{dataset}/{scene_id}']
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(intern_src))
    print(result.stdout[-1500:])
    if result.returncode != 0:
        print('STDERR:')
        print(result.stderr[-1500:])
    if not glb_path.exists():
        raise FileNotFoundError(f'Composition did not produce {glb_path}')
    return glb_path


def convert_scene_id_to_usd(scene_id: str, dataset: str = 'scannet', output_root: Optional[Path] = None) -> Tuple[Path, bool, str]:
    """Convert one composed InternScenes GLB into an Isaac Sim-compatible USD file."""
    project_root = PROJECT_ROOT
    intern_src = project_root / 'external-lib' / 'InternScenes'
    isaac_py = Path('/home/snt/isaacsim/python.sh')

    if output_root is None:
        output_root = project_root / 'issacsim-assets'

    ensure_layout_for_scene(scene_id, dataset)
    download_scene_assets_for_id(scene_id, dataset)
    glb_path = compose_scene_id(scene_id, dataset)

    usd_path = output_root / scene_id / 'usd' / 'scene.usd'
    usd_path.parent.mkdir(parents=True, exist_ok=True)

    cmd = [
        str(isaac_py),
        str(intern_src / 'glb2usd_headless.py'),
        '--file', str(glb_path),
        '--out', str(usd_path),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    ok = result.returncode == 0 and usd_path.exists()
    print(f'[{scene_id}] rc={result.returncode}, usd_exists={usd_path.exists()}')
    if not ok:
        print('STDOUT:')
        print(result.stdout[-1500:])
        print('STDERR:')
        print(result.stderr[-1500:])
    return usd_path, ok, (result.stderr.strip() if result.stderr else 'ok')


In [4]:
import pandas as pd

scene_ids = ['scene0004_00']  # add more scene ids here
summary = []

for sid in scene_ids:
    try:
        usd_path, ok, msg = convert_scene_id_to_usd(sid, dataset='scannet')
        summary.append({
            'scene_id': sid,
            'usd_path': str(usd_path),
            'success': ok,
            'message': msg,
        })
    except Exception as e:
        summary.append({
            'scene_id': sid,
            'usd_path': None,
            'success': False,
            'message': str(e),
        })

summary_df = pd.DataFrame(summary)
print(summary_df)

summary_path = PROJECT_ROOT / 'issacsim-assets' / 'scene_conversion_summary.csv'
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(summary_path, index=False)
print('saved conversion summary ->', summary_path)


scene: /home/snt/projects/AgenticMemoryNav/data/Layout_info/scannet/scene0004_00/layout.json
objects: 12  unique assets: 10
per-object GLBs: 2  archives: 1
downloaded into /home/snt/projects/AgenticMemoryNav/data

process_instance 1 done
process_instance 2 done
process_instance 4 done
process_instance 9 done
process_instance 0 done
process_instance 5 done
process_instance 7 done
process_instance 3 doneprocess_instance 6 done

process_instance 8 done
process_instance 10 done
process_instance 11 done
Composed glb scene has been saved to /home/snt/projects/AgenticMemoryNav/tutorial/examples/composed_scenes/scannet/scene0004_00/glb_scene.glb
GLB_PATH=/home/snt/projects/AgenticMemoryNav/tutorial/examples/composed_scenes/scannet/scene0004_00/glb_scene.glb
GLB_EXISTS=True

[scene0004_00] rc=0, usd_exists=True
       scene_id                                           usd_path  success  \
0  scene0004_00  /home/snt/projects/AgenticMemoryNav/issacsim-a...     True   

                           

## Render a perspective preview

Quick visual sanity check of the converted USD using Isaac Sim's perspective viewport.

In [10]:
from pathlib import Path
import subprocess

SCENE_ID = scene_ids[0]  # pick the scene converted above to preview
USD_OUT_PATH = PROJECT_ROOT / 'issacsim-assets' / SCENE_ID / 'usd' / 'scene.usd'
RENDER_OUT = PROJECT_ROOT / 'issacsim-assets' / SCENE_ID / 'render' / 'perspective.png'
RENDER_SCRIPT = PROJECT_ROOT / 'issacsim-assets' / SCENE_ID / 'render' / 'render_perspective.py'
RENDER_OUT.parent.mkdir(parents=True, exist_ok=True)

if not USD_OUT_PATH.exists():
    raise FileNotFoundError(f'USD file not found; convert {SCENE_ID} first: {USD_OUT_PATH}')
if not ISAAC_PY.exists():
    raise FileNotFoundError(f'Isaac Sim python not found: {ISAAC_PY}')

render_code = f'''
from pathlib import Path
import numpy as np

from isaacsim import SimulationApp

usd_path = Path({str(USD_OUT_PATH)!r})
out_path = Path({str(RENDER_OUT)!r})
out_path.parent.mkdir(parents=True, exist_ok=True)

app = SimulationApp({{"headless": True, "width": 1280, "height": 720}})
try:
    from omni.usd import get_context
    from pxr import Gf, Sdf, Usd, UsdGeom, UsdLux, UsdShade

    ctx = get_context()
    if not ctx.open_stage(str(usd_path)):
        raise RuntimeError(f'Isaac Sim failed to open USD: {{usd_path}}')
    while ctx.get_stage_loading_status()[2] > 0:
        app.update()

    stage = ctx.get_stage()
    root = stage.GetPrimAtPath('/World')
    if not root.IsValid():
        root = stage.GetDefaultPrim()
    if not root.IsValid():
        root = stage.GetPseudoRoot()

    bbox_cache = UsdGeom.BBoxCache(Usd.TimeCode.Default(), [UsdGeom.Tokens.default_, UsdGeom.Tokens.render], useExtentsHint=True)
    aligned = bbox_cache.ComputeWorldBound(root).ComputeAlignedRange()
    lower = np.array(aligned.GetMin(), dtype=np.float64)
    upper = np.array(aligned.GetMax(), dtype=np.float64)
    if (not np.isfinite(lower).all()) or (not np.isfinite(upper).all()) or np.any(upper <= lower):
        raise RuntimeError(f'Invalid scene bounds: lower={{lower.tolist()}}, upper={{upper.tolist()}}')

    center = (lower + upper) * 0.5
    extent = upper - lower
    radius = max(float(np.linalg.norm(extent) * 0.5), 1.0)
    up_axis = str(UsdGeom.GetStageUpAxis(stage)).lower()
    if up_axis == 'y':
        up = np.array([0.0, 1.0, 0.0], dtype=np.float64)
        direction = np.array([1.0, 0.55, 1.0], dtype=np.float64)
    else:
        up = np.array([0.0, 0.0, 1.0], dtype=np.float64)
        direction = np.array([1.0, -1.0, 0.65], dtype=np.float64)
    direction = direction / np.linalg.norm(direction)
    eye = center + direction * radius * 2.2

    material = UsdShade.Material.Define(stage, Sdf.Path('/World/preview_override_material'))
    shader = UsdShade.Shader.Define(stage, Sdf.Path('/World/preview_override_material/PreviewSurface'))
    shader.CreateIdAttr('UsdPreviewSurface')
    shader.CreateInput('diffuseColor', Sdf.ValueTypeNames.Color3f).Set(Gf.Vec3f(0.62, 0.64, 0.60))
    shader.CreateInput('roughness', Sdf.ValueTypeNames.Float).Set(0.45)
    material.CreateSurfaceOutput().ConnectToSource(shader.ConnectableAPI(), 'surface')
    UsdShade.MaterialBindingAPI(root).Bind(material, bindingStrength=UsdShade.Tokens.strongerThanDescendants)

    dome = UsdLux.DomeLight.Define(stage, Sdf.Path('/World/preview_dome_light'))
    dome.CreateIntensityAttr(1500.0)
    distant = UsdLux.DistantLight.Define(stage, Sdf.Path('/World/preview_key_light'))
    distant.CreateIntensityAttr(6500.0)
    distant.CreateAngleAttr(0.6)

    camera_path = '/World/perspective_preview_camera'
    camera = UsdGeom.Camera.Define(stage, Sdf.Path(camera_path))
    camera.GetFocalLengthAttr().Set(18.0)
    camera.GetHorizontalApertureAttr().Set(20.955)
    camera.GetVerticalApertureAttr().Set(15.2908)
    camera.GetClippingRangeAttr().Set(Gf.Vec2f(0.01, max(1000.0, radius * 10.0)))

    look_at = Gf.Matrix4d().SetLookAt(
        Gf.Vec3d(float(eye[0]), float(eye[1]), float(eye[2])),
        Gf.Vec3d(float(center[0]), float(center[1]), float(center[2])),
        Gf.Vec3d(float(up[0]), float(up[1]), float(up[2])),
    )
    xform = UsdGeom.Xformable(camera.GetPrim())
    xform.ClearXformOpOrder()
    xform.AddTransformOp().Set(look_at.GetInverse())

    light_xform = UsdGeom.Xformable(distant.GetPrim())
    light_xform.ClearXformOpOrder()
    light_xform.AddTransformOp().Set(look_at.GetInverse())

    from omni.kit.viewport.utility import capture_viewport_to_file, get_active_viewport
    from PIL import Image

    viewport = get_active_viewport()
    if viewport is None:
        raise RuntimeError('No active Isaac Sim viewport is available')
    viewport.camera_path = camera_path
    if hasattr(viewport, 'resolution'):
        viewport.resolution = (1280, 720)

    def capture(mode: str | None = None) -> tuple[float, float]:
        if mode is not None:
            try:
                viewport.render_mode = mode
            except Exception as error:
                print(f'Could not set render_mode={{mode!r}}: {{error}}')
        for _ in range(45):
            app.update()
        capture_viewport_to_file(viewport, str(out_path))
        for _ in range(60):
            app.update()
        image = np.asarray(Image.open(out_path).convert('RGB'), dtype=np.float32)
        return float(image.mean()), float(image.std())

    mean, std = capture('RayTracedLighting')
    final_mode = getattr(viewport, 'render_mode', 'unknown')
    if mean < 2.0 or std < 2.0:
        mean, std = capture('Wireframe')
        final_mode = getattr(viewport, 'render_mode', 'unknown')

    if not out_path.exists() or out_path.stat().st_size == 0:
        raise RuntimeError(f'Viewport capture did not write a valid image: {{out_path}}')

    print('stage_up_axis =', up_axis)
    print('scene_bounds_min =', lower.tolist())
    print('scene_bounds_max =', upper.tolist())
    print('camera_eye =', eye.tolist())
    print('camera_target =', center.tolist())
    print('render_mode =', final_mode)
    print('pixel_mean =', mean, 'pixel_std =', std)
    print('saved =', out_path)
finally:
    app.close()
'''

RENDER_SCRIPT.write_text(render_code, encoding='utf-8')
print('Rendering perspective viewport for', SCENE_ID)
print('USD input =', USD_OUT_PATH)
print('output PNG =', RENDER_OUT)

result = subprocess.run([str(ISAAC_PY), str(RENDER_SCRIPT)], capture_output=True, text=True)
print(result.stdout[-4000:])
if result.returncode != 0:
    print('STDERR:')
    print(result.stderr[-4000:])
    raise RuntimeError(f'Isaac Sim perspective render failed with rc={result.returncode}')

print('PNG exists =', RENDER_OUT.exists(), 'size_bytes =', RENDER_OUT.stat().st_size if RENDER_OUT.exists() else 0)


Rendering perspective viewport for scene0004_00
USD input = /home/snt/projects/AgenticMemoryNav/issacsim-assets/scene0004_00/usd/scene.usd
output PNG = /home/snt/projects/AgenticMemoryNav/issacsim-assets/scene0004_00/render/perspective.png

Ensure that the `SimulationApp` class is instantiated before importing
any other Omniverse/Isaac Sim modules, as shown below:

    ------------------------------------------------------------------
    from isaacsim import SimulationApp

    # instantiate the SimulationApp helper class
    simulation_app = SimulationApp({"headless": False})

    # execute other Omniverse/Isaac Sim imports after instantiating it
    from isaacsim...
    ------------------------------------------------------------------


[11.171s] [ext: isaacsim.simulation_app-2.18.4] startup
[11.183s] [ext: omni.anim.curve.core-1.6.0] startup
[11.202s] [ext: omni.graph.ui_nodes-2.11.1] startup
[11.255s] [ext: omni.hydra.scene_api-0.1.3] startup
[11.263s] [ext: omni.kit.manipulator.c